In [ ]:
!pip install catboost
!pip install lime

In [ ]:
# =============================================================================
# HYPERPARAMETER TUNING
# =============================================================================

param_grids = {

    # ── Logistic Regression ───────────────────────────────────────────────────
    "Logistic Regression": {
        "clf__C"        : [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
        "clf__solver"   : ["lbfgs"],
        "clf__max_iter" : [500, 1000, 2000],
    },

    # ── Decision Tree ─────────────────────────────────────────────────────────
    "Decision Tree": {
        "clf__max_depth"        : [4, 6, 8, 10, 12, 15, None],
        "clf__min_samples_leaf" : [10, 20, 30, 50],
        "clf__min_samples_split": [10, 20, 40, 60],
        "clf__criterion"        : ["gini", "entropy"],
    },

    # ── Random Forest ─────────────────────────────────────────────────────────
    "Random Forest": {
        "clf__n_estimators"     : [100, 200, 300],
        "clf__max_depth"        : [8, 10, 12, 15, None],
        "clf__min_samples_leaf" : [10, 20, 30],
        "clf__min_samples_split": [10, 20, 40],
        "clf__max_features"     : ["sqrt", "log2"],
    },

    # ── XGBoost ───────────────────────────────────────────────────────────────
    # Note: XGBoost does not accept class_weight parameter in multiclass mode.
    # Class imbalance is handled via sample_weight at fit time .
    "XGBoost": {
        "clf__n_estimators"     : [200, 300, 500],
        "clf__learning_rate"    : [0.01, 0.03, 0.05, 0.1],
        "clf__max_depth"        : [4, 6, 8],
        "clf__subsample"        : [0.7, 0.8, 1.0],
        "clf__colsample_bytree" : [0.7, 0.8, 1.0],
        "clf__reg_alpha"        : [0, 0.1, 0.5],
        "clf__reg_lambda"       : [1.0, 2.0, 5.0],
    },

    # ── LightGBM ──────────────────────────────────────────────────────────────
    "LightGBM": {
        "clf__n_estimators"      : [200, 300, 500],
        "clf__learning_rate"     : [0.01, 0.03, 0.05, 0.1],
        "clf__num_leaves"        : [31, 63, 127],
        "clf__max_depth"         : [6, 8, 10, -1],
        "clf__reg_alpha"         : [0, 0.1, 0.5],
        "clf__reg_lambda"        : [1.0, 2.0, 5.0],
        "clf__min_child_samples" : [20, 30, 50],
    },

    # ── CatBoost ──────────────────────────────────────────────────────────────
    "CatBoost": {
        "clf__iterations"          : [200, 300, 500],
        "clf__learning_rate"       : [0.01, 0.03, 0.05, 0.1],
        "clf__depth"               : [4, 6, 8],
        "clf__l2_leaf_reg"         : [1, 3, 5, 7],
        "clf__bagging_temperature" : [0.0, 0.5, 1.0],
    },

    # ── SVM (CalibratedClassifierCV wrapping LinearSVC) ───────────────────────
    # Parameters must be prefixed with clf__estimator__ because LinearSVC
    # is nested inside CalibratedClassifierCV as its estimator attribute
    "SVM": {
        "clf__estimator__C"        : [0.01, 0.1, 1.0, 5.0, 10.0],
        "clf__estimator__max_iter" : [1000, 2000, 3000],
    },
}

# =============================================================================
# ITERATION CONTROL
# Higher iterations improve tuning quality but increase runtime.
# =============================================================================

n_iter_map = {
    "Logistic Regression" : 5,
    "Decision Tree"       : 10,
    "Random Forest"       : 10,
    "XGBoost"             : 15,
    "LightGBM"            : 15,
    "CatBoost"            : 15,
    "SVM"                 : 5,
}

# =============================================================================
# RANDOMIZED SEARCH TUNING LOOP
# =============================================================================

print("\n")
print("=" * 90)
print("  HYPERPARAMETER TUNING — TASK 4: ORDER PROCESSING DELAY CLASSIFICATION")
print("=" * 90)

tuned_results = {}

for model_name, pipeline in models.items():

    print(f"\n   Tuning → {model_name} ...", end=" ", flush=True)
    t0 = time()

    rs = RandomizedSearchCV(
        estimator           = pipeline,
        param_distributions = param_grids[model_name],
        n_iter              = n_iter_map[model_name],
        cv                  = skf,
        # f1_macro — equal weight to all three processing classes
        scoring             = "f1_macro",
        n_jobs              = N_JOBS,
        random_state        = RANDOM_STATE,
        verbose             = 0,
        refit               = True,
        return_train_score  = False,
    )

    # XGBoost requires sample_weight at fit time
    # All other models use class_weight="balanced" internally
    if model_name == "XGBoost":
        rs.fit(X_train, y_train, clf__sample_weight=sample_weights)
    else:
        rs.fit(X_train, y_train)

    train_time = time() - t0

    best_pipeline_tuned = rs.best_estimator_
    y_pred              = best_pipeline_tuned.predict(X_test)
    y_prob              = best_pipeline_tuned.predict_proba(X_test)

    # ── Metrics  ─────────────
    acc         = accuracy_score(y_test, y_pred)
    macro_prec  = precision_score(y_test, y_pred, average="macro",    zero_division=0)
    macro_rec   = recall_score(y_test, y_pred,    average="macro",    zero_division=0)
    macro_f1    = f1_score(y_test, y_pred,        average="macro",    zero_division=0)
    weighted_f1 = f1_score(y_test, y_pred,        average="weighted", zero_division=0)
    per_cls_f1  = f1_score(y_test, y_pred,        average=None,       zero_division=0)

    # Multiclass ROC-AUC — One-vs-Rest, macro averaging
    roc_auc = roc_auc_score(
        y_test, y_prob,
        multi_class="ovr",
        average="macro",
        labels=classes
    )

    tuned_results[model_name] = {
        "Accuracy"         : round(acc,           4),
        "Macro Precision"  : round(macro_prec,    4),
        "Macro Recall"     : round(macro_rec,     4),
        "Macro F1"         : round(macro_f1,      4),
        "Weighted F1"      : round(weighted_f1,   4),
        "F1 Fast"          : round(per_cls_f1[0], 4),
        "F1 Normal"        : round(per_cls_f1[1], 4),
        "F1 Delayed"       : round(per_cls_f1[2], 4),
        "ROC-AUC"          : round(roc_auc,       4),
        "Best Params"      : rs.best_params_,
        "Train Time (s)"   : round(train_time,    1),
        "_pipeline"        : best_pipeline_tuned,
        "_y_pred"          : y_pred,
        "_y_prob"          : y_prob,
    }

    print(
        f"done [{train_time:.1f}s]  "
        f"MacroF1={macro_f1:.4f}  "
        f"ROC-AUC={roc_auc:.4f}"
    )

# =============================================================================
# POST-TUNING COMPARISON TABLE
# =============================================================================

print("\n")
print("=" * 110)
print("  POST-TUNING MODEL COMPARISON — TASK 4: ORDER PROCESSING DELAY CLASSIFICATION")
print("=" * 110)

metrics_cols_tuned = [
    "Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1",
    "F1 Fast", "F1 Normal", "F1 Delayed",
    "ROC-AUC", "Train Time (s)"
]

tuned_compare_df = pd.DataFrame(
    {m: {k: tuned_results[m][k] for k in metrics_cols_tuned} for m in tuned_results}
).T.reset_index().rename(columns={"index": "Model"})

tuned_compare_df = tuned_compare_df.sort_values(
    "Macro F1", ascending=False
).reset_index(drop=True)
tuned_compare_df.insert(0, "Rank", tuned_compare_df.index + 1)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4f}".format)

print(tuned_compare_df.to_string(index=False))
print("=" * 110)

best_tuned_name     = tuned_compare_df.iloc[0]["Model"]
best_tuned_pipeline = tuned_results[best_tuned_name]["_pipeline"]
best_tuned_y_pred   = tuned_results[best_tuned_name]["_y_pred"]
best_tuned_y_prob   = tuned_results[best_tuned_name]["_y_prob"]

print(f"\n   Best Tuned Model  : {best_tuned_name}")
print(f"   Accuracy          : {tuned_results[best_tuned_name]['Accuracy']:.4f}")
print(f"   Macro Precision   : {tuned_results[best_tuned_name]['Macro Precision']:.4f}")
print(f"   Macro Recall      : {tuned_results[best_tuned_name]['Macro Recall']:.4f}")
print(f"   Macro F1          : {tuned_results[best_tuned_name]['Macro F1']:.4f}")
print(f"   Weighted F1       : {tuned_results[best_tuned_name]['Weighted F1']:.4f}")
print(f"   F1 Fast           : {tuned_results[best_tuned_name]['F1 Fast']:.4f}")
print(f"   F1 Normal         : {tuned_results[best_tuned_name]['F1 Normal']:.4f}")
print(f"   F1 Delayed        : {tuned_results[best_tuned_name]['F1 Delayed']:.4f}")
print(f"   ROC-AUC           : {tuned_results[best_tuned_name]['ROC-AUC']:.4f}")
print(f"   Best Params       : {tuned_results[best_tuned_name]['Best Params']}")
print("=" * 110)

tuned_compare_df.to_csv(
    "outputs_task4/task4_tuned_model_comparison.csv", index=False
)
print("[OK] Saved → outputs_task4/task4_tuned_model_comparison.csv")

# =============================================================================
# BEST HYPERPARAMETERS — ALL MODELS
# =============================================================================

print("\n── Best Hyperparameters — All Models ───────────────────────────────────")
for m in tuned_results:
    print(f"\n   {m}:")
    for param, val in tuned_results[m]["Best Params"].items():
        print(f"     {param:<40} : {val}")

params_rows = [
    {"Model": m, "Best Params": str(tuned_results[m]["Best Params"])}
    for m in tuned_results
]
pd.DataFrame(params_rows).to_csv(
    "outputs_task4/task4_best_params.csv", index=False
)
print("\n[OK] Saved → outputs_task4/task4_best_params.csv")

# =============================================================================
# CLASSIFICATION REPORT — BEST TUNED MODEL
# =============================================================================

print(f"\n── Classification Report — {best_tuned_name} (Tuned) ──────────────────")
print(classification_report(
    y_test, best_tuned_y_pred,
    target_names=CLASS_NAMES,
    digits=4
))

# =============================================================================
# VISUALISATIONS — POST TUNING
# =============================================================================

print("\n── Post-Tuning Visualisations ──────────────────────────────────────────")

PALETTE_T          = sns.color_palette("Set2", n_colors=len(tuned_results))
MODEL_ORDER_TUNED  = tuned_compare_df["Model"].tolist()

# ── Core metrics grouped bar chart ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
bar_metrics_t = ["Accuracy", "Macro Precision", "Macro Recall",
                 "Macro F1", "Weighted F1"]
x     = np.arange(len(MODEL_ORDER_TUNED))
width = 0.15

for i, metric in enumerate(bar_metrics_t):
    vals = [tuned_results[m][metric] for m in MODEL_ORDER_TUNED]
    bars = ax.bar(x + i * width, vals, width, label=metric, color=PALETTE_T[i])
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=6, rotation=90
        )

ax.set_xticks(x + width * 2)
ax.set_xticklabels(MODEL_ORDER_TUNED, rotation=25, ha="right", fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Task 4 — Tuned Model Comparison: Core Metrics",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs_task4/task4_tuned_model_comparison_bars.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("[OK] Saved → outputs_task4/task4_tuned_model_comparison_bars.png")

# ── Per-class F1 grouped bar chart ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
per_class_metrics_t = ["F1 Fast", "F1 Normal", "F1 Delayed"]
per_class_colors_t  = ["#43A047", "#FFA726", "#EF5350"]
x2     = np.arange(len(MODEL_ORDER_TUNED))
width2 = 0.25

for i, (metric, color) in enumerate(
    zip(per_class_metrics_t, per_class_colors_t)
):
    vals = [tuned_results[m][metric] for m in MODEL_ORDER_TUNED]
    bars = ax.bar(x2 + i * width2, vals, width2,
                  label=metric, color=color, alpha=0.85)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=7, rotation=90
        )

ax.set_xticks(x2 + width2)
ax.set_xticklabels(MODEL_ORDER_TUNED, rotation=25, ha="right", fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("F1-Score")
ax.set_title(
    "Task 4 — Per-Class F1 Score: Tuned Models (Fast | Normal | Delayed)",
    fontsize=12, fontweight="bold"
)
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs_task4/task4_tuned_per_class_f1.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("[OK] Saved → outputs_task4/task4_tuned_per_class_f1.png")

# ── Heatmap — post-tuning metrics ─────────────────────────────────────────────
hm_cols_t = [
    "Accuracy", "Macro F1", "Weighted F1",
    "F1 Fast", "F1 Normal", "F1 Delayed", "ROC-AUC"
]
hm_df_t = tuned_compare_df.set_index("Model")[hm_cols_t].astype(float)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    hm_df_t,
    annot=True, fmt=".4f", cmap="YlGn",
    linewidths=0.5, ax=ax,
    annot_kws={"size": 8.5}
)
ax.set_title(
    "Task 4 — Tuned Model Performance Heatmap",
    fontsize=13, fontweight="bold"
)
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("outputs_task4/task4_tuned_heatmap.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("[OK] Saved → outputs_task4/task4_tuned_heatmap.png")

# ── Confusion matrix — best tuned model ───────────────────────────────────────
cm_t = confusion_matrix(y_test, best_tuned_y_pred)
disp_t = ConfusionMatrixDisplay(
    confusion_matrix=cm_t, display_labels=CLASS_NAMES
)
fig, ax = plt.subplots(figsize=(6, 5))
disp_t.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(
    f"Confusion Matrix — {best_tuned_name} (Tuned)",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig("outputs_task4/task4_tuned_confusion_matrix.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("[OK] Saved → outputs_task4/task4_tuned_confusion_matrix.png")

# ── Confusion matrices grid — all tuned models ────────────────────────────────
n_models_t = len(MODEL_ORDER_TUNED)
ncols_t    = 4
nrows_t    = (n_models_t + ncols_t - 1) // ncols_t

fig, axes = plt.subplots(nrows_t, ncols_t,
                          figsize=(ncols_t * 4.5, nrows_t * 4))
axes = axes.flatten()

for i, model_name in enumerate(MODEL_ORDER_TUNED):
    cm_i   = confusion_matrix(y_test, tuned_results[model_name]["_y_pred"])
    disp_i = ConfusionMatrixDisplay(
        confusion_matrix=cm_i,
        display_labels=["Fast", "Normal", "Delayed"]
    )
    disp_i.plot(ax=axes[i], colorbar=False, cmap="Blues")
    axes[i].set_title(
        f"{model_name}\nMacro F1={tuned_results[model_name]['Macro F1']:.4f}",
        fontsize=9, fontweight="bold"
    )
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    "Task 4 — Confusion Matrices: All Tuned Models",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig("outputs_task4/task4_tuned_confusion_matrices_all.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("[OK] Saved → outputs_task4/task4_tuned_confusion_matrices_all.png")

# ── ROC curves — one panel per class (OvR), all tuned models ──────────────────
y_test_bin_t  = label_binarize(y_test, classes=[0, 1, 2])
proc_labels_t = ["Fast (0-2d)", "Normal (3d)", "Delayed (4-6d)"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for class_idx, class_name in enumerate(proc_labels_t):
    ax = axes[class_idx]
    for i, model_name in enumerate(MODEL_ORDER_TUNED):
        y_prob_model = tuned_results[model_name]["_y_prob"]
        fpr, tpr, _  = roc_curve(
            y_test_bin_t[:, class_idx], y_prob_model[:, class_idx]
        )
        roc_auc_c = auc(fpr, tpr)
        ax.plot(
            fpr, tpr,
            label=f"{model_name} ({roc_auc_c:.3f})",
            color=PALETTE_T[i], lw=1.6
        )
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_title(f"OvR — {class_name}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=6.5, loc="lower right")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("True Positive Rate")
fig.suptitle(
    "Task 4 — ROC Curves (One-vs-Rest): All Tuned Models",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig("outputs_task4/task4_tuned_roc_curves.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("[OK] Saved → outputs_task4/task4_tuned_roc_curves.png")

# ── CV Macro F1 bar chart — tuned models ──────────────────────────────────────
# Re-run CV on tuned pipelines to confirm post-tuning generalisation
print("\n── Post-Tuning Cross-Validation Macro F1 (5-Fold) ─────────────────────")

tuned_cv_means = []
tuned_cv_stds  = []

for model_name in MODEL_ORDER_TUNED:
    cv_s = cross_val_score(
        tuned_results[model_name]["_pipeline"],
        X_train, y_train,
        cv=skf, scoring="f1_macro", n_jobs=N_JOBS
    )
    tuned_cv_means.append(cv_s.mean())
    tuned_cv_stds.append(cv_s.std())
    print(f"   {model_name:<22} CV Macro F1 = {cv_s.mean():.4f} ± {cv_s.std():.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(
    MODEL_ORDER_TUNED, tuned_cv_means,
    xerr=tuned_cv_stds,
    color=PALETTE_T[:len(MODEL_ORDER_TUNED)],
    height=0.55, capsize=5, ecolor="gray"
)
ax.set_xlabel("CV Macro F1-Score (mean ± std)")
ax.set_title(
    "Task 4 — Cross-Validated Macro F1: Tuned Models (5-Fold)",
    fontsize=12, fontweight="bold"
)
ax.set_xlim(0, 1.05)
for i, (m, s) in enumerate(zip(tuned_cv_means, tuned_cv_stds)):
    ax.text(m + s + 0.01, i, f"{m:.4f}", va="center", fontsize=8)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs_task4/task4_tuned_cv_macro_f1.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("[OK] Saved → outputs_task4/task4_tuned_cv_macro_f1.png")

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n")
print("=" * 110)
print("  FINAL SUMMARY — TASK 4: POST-TUNING RESULTS")
print("=" * 110)
print(tuned_compare_df[[
    "Rank", "Model", "Accuracy",
    "Macro Precision", "Macro Recall",
    "Macro F1", "Weighted F1",
    "F1 Fast", "F1 Normal", "F1 Delayed",
    "ROC-AUC", "Train Time (s)"
]].to_string(index=False))
print("=" * 110)
print(f"\n   Best Tuned Model  : {best_tuned_name}")
print(f"   Accuracy          : {tuned_results[best_tuned_name]['Accuracy']:.4f}")
print(f"   Macro Precision   : {tuned_results[best_tuned_name]['Macro Precision']:.4f}")
print(f"   Macro Recall      : {tuned_results[best_tuned_name]['Macro Recall']:.4f}")
print(f"   Macro F1          : {tuned_results[best_tuned_name]['Macro F1']:.4f}")
print(f"   Weighted F1       : {tuned_results[best_tuned_name]['Weighted F1']:.4f}")
print(f"   F1 Fast           : {tuned_results[best_tuned_name]['F1 Fast']:.4f}")
print(f"   F1 Normal         : {tuned_results[best_tuned_name]['F1 Normal']:.4f}")
print(f"   F1 Delayed        : {tuned_results[best_tuned_name]['F1 Delayed']:.4f}")
print(f"   ROC-AUC           : {tuned_results[best_tuned_name]['ROC-AUC']:.4f}")
print(f"\n   Saved outputs:")
print("    • outputs_task4/task4_tuned_model_comparison.csv")
print("    • outputs_task4/task4_best_params.csv")
print("    • outputs_task4/task4_tuned_model_comparison_bars.png")
print("    • outputs_task4/task4_tuned_per_class_f1.png")
print("    • outputs_task4/task4_tuned_heatmap.png")
print("    • outputs_task4/task4_tuned_confusion_matrix.png")
print("    • outputs_task4/task4_tuned_confusion_matrices_all.png")
print("    • outputs_task4/task4_tuned_roc_curves.png")
print("    • outputs_task4/task4_tuned_cv_macro_f1.png")
print("=" * 110)